In [24]:
using LinearAlgebra
using DynamicPolynomials
using SumOfSquares
using MosekTools
using JuMP

using ppt2

include("utils.jl")
using .Utils

In [25]:
function mprint(x)
    show(stdout, MIME"text/plain"(), x)
end

mprint (generic function with 1 method)

In [37]:
# positivity test for generated forms

d = 3
del, v, V = gen_pncp(d, d)

form = del * v + 10 * vec(V * V')

Q = vec(poly2mat(form, d, d))

for _ in 1:1000000
    x = randn(d)
    y = randn(d)
    z = kron(x, y)

    eval = Q' * kron(z, z)
    if eval < -1e-8
        println("Found a negative value: ", eval)
        break
    end
end

In [39]:
# Copied from MATLAB

v = [
   -0.5060
    0.3052
   -0.8745
    1.1055
    0.0918
    0.2058
    0.0783
    0.0963
   -1.8706
    0.3052
    0.6757
   -0.2828
    0.2664
   -1.4281
    1.6131
    1.3889
   -0.4919
   -1.5970
   -0.8745
   -0.2828
    0.4998
   -1.7145
    0.6318
   -0.5592
    0.4722
    1.3449
    0.7394
    1.1055
    0.2664
   -1.7145
   -2.3953
   -0.6455
    0.1190
   -0.0082
   -1.6380
    1.0777
    0.0918
   -1.4281
    0.6318
   -0.6455
    1.5746
   -1.8027
   -0.0464
    0.7299
   -2.3220
    0.2058
    1.6131
   -0.5592
    0.1190
   -1.8027
    1.4639
   -1.7870
    1.5577
    0.1464
    0.0783
    1.3889
    0.4722
   -0.0082
   -0.0464
   -1.7870
   -0.1932
    1.1561
   -1.2673
    0.0963
   -0.4919
    1.3449
   -1.6380
    0.7299
    1.5577
    1.1561
   -1.1793
    0.4079
   -1.8706
   -1.5970
    0.7394
    1.0777
   -2.3220
    0.1464
   -1.2673
    0.4079
   -0.2600
];

V = [
   -0.5511    1.5360   -1.1597    1.1090
   -0.7599   -0.7417    0.3097    0.2356
    1.8175   -0.3787    1.3324   -1.3714
    2.5811   -2.0486    3.1058    0.7221
   -2.3082    1.0346   -0.4188   -2.1686
    0.2470   -2.2139    1.9240    1.0432
   -1.9265    0.3228   -0.8258    1.2775
    1.0580    0.0157    1.2487   -0.8869
   -1.3906   -1.2938    0.1407    0.3199
];

state = (1/3)*[
    1 0 0 0 1 0 0 0 1
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    1 0 0 0 1 0 0 0 1
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    1 0 0 0 1 0 0 0 1
];

In [49]:
form = v + vec(V * V')
M = poly2mat(form, 3, 3)
mprint(M)

9×9 Matrix{Float64}:
  4.73179   -0.513149  -5.52387   -6.26458     0.995081   1.24755   4.01024   -0.117841  -4.45182
 -0.513149   1.95469   -1.29349    0.995081   -1.08209    1.18497  -0.117841  -1.12975    3.3401
 -5.52387   -1.29349    7.60255    1.24755     1.18497    1.86102  -4.45182    3.3401    -1.5493
 -6.26458    0.995081   1.24755   18.631     -11.5893    12.0208   -7.28426    3.3041    -1.21338
  0.995081  -1.08209    1.18497  -11.5893     12.851     -7.73138   3.3041    -0.295557   1.02905
  1.24755    1.18497    1.86102   12.0208     -7.73138   11.2163   -1.21338    1.02905    3.27169
  4.01024   -0.117841  -4.45182   -7.28426     3.3041    -1.21338   5.93635   -3.04126    1.28653
 -0.117841  -1.12975    3.3401     3.3041     -0.295557   1.02905  -3.04126    2.28615   -1.19169
 -4.45182    3.3401    -1.5493    -1.21338     1.02905    3.27169   1.28653   -1.19169    3.46982

In [51]:
phi = mat2block(M, 3, 3)
bstate = mat2block(state, 3, 3)

mapped = apply_map(bstate, phi)
mprint(block2mat(mapped))

9×9 Matrix{Float64}:
  1.57726    -0.17105    -1.84129   -2.08819    0.331694    0.415851   1.33675    -0.0392805  -1.48394
 -0.17105     0.651563   -0.431165   0.331694  -0.360695    0.394991  -0.0392805  -0.376583    1.11337
 -1.84129    -0.431165    2.53418    0.415851   0.394991    0.62034   -1.48394     1.11337    -0.516432
 -2.08819     0.331694    0.415851   6.21032   -3.86311     4.00693   -2.42809     1.10137    -0.404459
  0.331694   -0.360695    0.394991  -3.86311    4.28367    -2.57713    1.10137    -0.0985189   0.343018
  0.415851    0.394991    0.62034    4.00693   -2.57713     3.73877   -0.404459    0.343018    1.09056
  1.33675    -0.0392805  -1.48394   -2.42809    1.10137    -0.404459   1.97878    -1.01375     0.428845
 -0.0392805  -0.376583    1.11337    1.10137   -0.0985189   0.343018  -1.01375     0.762051   -0.397232
 -1.48394     1.11337    -0.516432  -0.404459   0.343018    1.09056    0.428845   -0.397232    1.15661

In [98]:
function check_candidate(candidate_state, d)
    del, v, V = gen_pncp(d, d)
    form = del * v + 10 * vec(V * V')

    M = poly2mat(form, d, d)

    phi = mat2block(M, d, d)

    state = mat2block(candidate_state, d, d)

    mapped = block2mat(apply_map(state, phi))

    min_val = minimum(real(eigvals(mapped)))
    return min_val, del, v, V
end

function witness_search(candidate_state, d, attempts)
    for _ in 1:attempts
        min_val, del, v, V = check_candidate(candidate_state, d)
        if min_val < -1e-6
            println("Found a witness with minimum eigenvalue: ", min_val)
            return del, v, V
        end
    end
    println("No witness found after ", attempts, " attempts.")
    return nothing
end

witness_search (generic function with 1 method)

In [99]:
d = 3
candidate_state = ones(9, 9)

witness_search(candidate_state, d, 100)

No witness found after 100 attempts.


In [100]:
d = 3
candidate_state = Utils.sep_state(3, 3)

witness_search(candidate_state, d, 100)

No witness found after 100 attempts.


In [116]:
d = 3
candidate_state = Utils.C3()

witness_search(candidate_state, d, 100)

No witness found after 100 attempts.


In [102]:
d = 4
candidate_state = Utils.C4(0.5, 0.1)

witness_search(candidate_state, d, 10)

No witness found after 10 attempts.
